# Titanic - Machine Learning from Disaster

Kaggle 竞赛：预测铁达尼号乘客生存情况

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)

## 1. 数据加载与探索

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'\nSurvival rate: {train["Survived"].mean():.3f}')
print(f'\nMissing values (train):')
print(train.isnull().sum()[train.isnull().sum() > 0])
print(f'\nMissing values (test):')
print(test.isnull().sum()[test.isnull().sum() > 0])

Train shape: (891, 12)
Test shape: (418, 11)

Survival rate: 0.384

Missing values (train):
Age         177
Cabin       687
Embarked      2
dtype: int64

Missing values (test):
Age       86
Fare       1
Cabin    327
dtype: int64


In [3]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. 特征工程

In [4]:
def engineer_features(df):
    df = df.copy()
    
    # 从姓名中提取称谓
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    title_map = {
        'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
        'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', 'Don': 'Rare',
        'Dona': 'Rare', 'Lady': 'Rare', 'Countess': 'Rare',
        'Jonkheer': 'Rare', 'Sir': 'Rare', 'Capt': 'Rare'
    }
    df['Title'] = df['Title'].map(title_map).fillna('Rare')
    
    # 家庭规模
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # 从 Cabin 提取是否有舱位信息
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    
    # 填充缺失值 (必须在分箱之前)
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # 年龄填充 (按 Title 和 Pclass 分组)
    df['Age'] = df.groupby(['Title', 'Pclass'])['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Age'] = df['Age'].fillna(df['Age'].median())
    
    # 分箱
    df['FareBin'] = pd.qcut(df['Fare'], 4, labels=False, duplicates='drop')
    df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                          labels=[0, 1, 2, 3, 4])
    
    # 编码
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    
    embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', dtype=int)
    title_dummies = pd.get_dummies(df['Title'], prefix='Title', dtype=int)
    
    df = pd.concat([df, embarked_dummies, title_dummies], axis=1)
    
    # 最终检查：确保没有 NaN
    df = df.fillna(0)
    
    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)
print('Feature engineering complete')

Feature engineering complete


In [5]:
features = [
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'FamilySize', 'IsAlone', 'HasCabin', 'FareBin', 'AgeBin',
    'Embarked_C', 'Embarked_Q', 'Embarked_S',
    'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare'
]

X_train = train_fe[features]
y_train = train_fe['Survived']
X_test = test_fe[features]

print(f'Features: {len(features)}')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

Features: 19
X_train shape: (891, 19)
X_test shape: (418, 19)


## 3. 模型训练与评估

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=7, min_samples_split=6,
    min_samples_leaf=3, random_state=42
)
rf_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Random Forest CV: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})')

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.1,
    min_samples_split=6, min_samples_leaf=3, random_state=42
)
gb_scores = cross_val_score(gb, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Gradient Boosting CV: {gb_scores.mean():.4f} (+/- {gb_scores.std():.4f})')

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Logistic Regression CV: {lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})')

Random Forest CV: 0.8305 (+/- 0.0133)


Gradient Boosting CV: 0.8327 (+/- 0.0133)
Logistic Regression CV: 0.8350 (+/- 0.0112)


## 4. 生成提交文件

In [7]:
# 使用表现最好的模型 (Logistic Regression CV 最高)
best_model = lr
best_model.fit(X_train, y_train)

predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
submission.to_csv('../submission.csv', index=False)
print(f'Submission saved: {submission.shape}')
submission.head()

Submission saved: (418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
